# Full Pix2Pix Synthetic X-ray Workflow

This notebook runs the full process around your Pix2Pix synthetic generation pipeline:

1. Check project paths and dependencies
2. Inspect COCO labels
3. Build an object cutout library from COCO annotations
4. Prepare tray masks / matched empty trays when needed
5. Optionally build/train a Pix2Pix dataset
6. Generate synthetic scenes with `generate_pix2pixV2_MAHADIST.py`
7. Compare generated images against held-out real images
8. Plot, preview, and launch the dashboard

The dashboard is still best for interactive inspection. This notebook is the reproducible process record.

## 0. Run controls

Set the stage toggles before running all cells. Expensive or GUI steps are off by default.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
from datetime import datetime

# Safety switch. Leave True to print commands first. Set False when you want cells to execute commands.
DRY_RUN = True

# Optional preparation stages.
RUN_OBJECT_LIBRARY = False
RUN_MATCH_EMPTY_TRAYS = False
RUN_TRAY_MASK_GUI = False
RUN_BUILD_AB_DATASET = False
RUN_TRAIN_PIX2PIX = False

# Main generation/evaluation stages.
RUN_GENERATE_SCENES = True
RUN_REAL_VS_GENERATED_REPORT = True

def run_cmd(cmd, *, cwd=None, run=True, check=True):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    if DRY_RUN or not run:
        print("DRY_RUN: command not executed")
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

def count_files(folder, suffixes=(".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff")):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in suffixes)

## 1. Configure project paths

These defaults match the commands already documented inside your Pix2Pix scripts. Change only the paths for a new run.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Pix2Pix":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

PIX2PIX_NOTEBOOK_DIR = PROJECT_ROOT / "Codes_Notebooks/Pix2Pix"
PIX2PIX_EXTERNAL_DIR = PROJECT_ROOT / "external/pix2pix"

# Raw annotation/image source for shampoo cutouts.
RAW_IMAGES_DIR = PROJECT_ROOT / "data/raw/Shampoo"
COCO_JSON = RAW_IMAGES_DIR / "result.json"
COLOR_PALETTE_JSON = RAW_IMAGES_DIR / "color_palette.json"
OBJECT_LIBRARY_DIR = RAW_IMAGES_DIR / "Cropped_Library"

# Real aligned Pix2Pix AB dataset and mask folders.
REAL_AB_TRAIN_DIR = PROJECT_ROOT / "datasets/SHAMPOOBLADEWITHTRAY_COMPLETE/train"
REAL_AB_TEST_DIR = PROJECT_ROOT / "datasets/SHAMPOOBLADEWITHTRAY_COMPLETE/test"
REAL_EVAL_DIR = REAL_AB_TEST_DIR
REAL_REF_DIR = REAL_AB_TRAIN_DIR

TRAY_MASK_TRAIN_DIR = PROJECT_ROOT / "datasets/SHAMPOOBLADEWITHTRAY_COMPLETE/matched_masks/train/tray"
TRAY_MASK_TEST_DIR = PROJECT_ROOT / "datasets/SHAMPOOBLADEWITHTRAY_COMPLETE/matched_masks/test/tray"
BLADE_MASK_TRAIN_DIR = PROJECT_ROOT / "datasets/SHAMPOOBLADEWITHTRAY_COMPLETE/matched_masks/train/blade"

# Optional empty-tray preparation paths.
EMPTY_TRAY_DIR = PROJECT_ROOT / "data/interim/GAN/Empty"

# Pix2Pix checkpoint and output run.
MODEL_NAME = "Shampoo_NOBGR_pix2pix_StructCond_V1_Stage24_COMPLETESyn"
OUT_DATASET = PROJECT_ROOT / "results/_gen_stage23_combo_tray"
GENERATED_DIR = OUT_DATASET / "generated"
GENERATED_SUMMARY_JSON = GENERATED_DIR / "generated_combo_summary.json"
REAL_VS_GENERATED_JSON = GENERATED_DIR / "real_vs_generated_report.json"

RUN_CONFIG = {
    "generate_mode": "combo",
    "combo_mode": "stage22_random",
    "classes": "Shampoo",
    "count": "1",
    "num_scenes": 200,
    "seed": 12,
    "epoch": "latest",
    "canvas_h": 1024,
    "canvas_w": 1024,
    "rand_scale_min": 0.95,
    "rand_scale_max": 0.95,
    "rand_rot_min": -5,
    "rand_rot_max": 5,
    "max_horizontal_shift": 150,
    "tray_horizontal_margin_px": 45,
    "tray_vertical_margin_px": 0,
    "max_vertical_shift": 8,
    "x_search_step": 12,
    "max_transform_candidates": 8,
    "mahal_pca_dim": 32,
}

paths = {
    "project_root": PROJECT_ROOT,
    "pix2pix_external_dir": PIX2PIX_EXTERNAL_DIR,
    "raw_images_dir": RAW_IMAGES_DIR,
    "coco_json": COCO_JSON,
    "color_palette_json": COLOR_PALETTE_JSON,
    "object_library_dir": OBJECT_LIBRARY_DIR,
    "real_ab_train_dir": REAL_AB_TRAIN_DIR,
    "real_ab_test_dir": REAL_AB_TEST_DIR,
    "tray_mask_train_dir": TRAY_MASK_TRAIN_DIR,
    "blade_mask_train_dir": BLADE_MASK_TRAIN_DIR,
    "out_dataset": OUT_DATASET,
    "generated_summary_json": GENERATED_SUMMARY_JSON,
}

for name, path in paths.items():
    print(f"{name:24s}: {path} | exists={path.exists()}")

## 2. Quick dependency and data sanity checks

In [ ]:
required_modules = ["cv2", "numpy", "PIL", "torch", "torchvision", "sklearn", "pandas", "matplotlib"]
missing = []
for mod in required_modules:
    try:
        __import__(mod)
    except Exception as exc:
        missing.append((mod, str(exc)))

if missing:
    print("Missing or failing imports:")
    for mod, reason in missing:
        print(f"  {mod}: {reason}")
else:
    print("All checked modules imported successfully.")

print("\nImage counts:")
for label, folder in [
    ("real train AB", REAL_AB_TRAIN_DIR),
    ("real test AB", REAL_AB_TEST_DIR),
    ("tray masks train", TRAY_MASK_TRAIN_DIR),
    ("blade masks train", BLADE_MASK_TRAIN_DIR),
    ("generated", GENERATED_DIR),
]:
    print(f"{label:18s}: {count_files(folder):5d} | {folder}")

## 3. Inspect COCO categories

This confirms the class names available to `--classes` and the palette mapping used for semantic masks.

In [ ]:
if COCO_JSON.exists():
    coco = json.loads(COCO_JSON.read_text())
    categories = sorted(coco.get("categories", []), key=lambda c: c.get("id", 0))
    print("COCO categories:")
    for c in categories:
        print(f"  id={c.get('id'):>3} | name={c.get('name')}")
else:
    print(f"COCO JSON not found: {COCO_JSON}")

if COLOR_PALETTE_JSON.exists():
    print("\nPalette mapping:")
    print(json.dumps(json.loads(COLOR_PALETTE_JSON.read_text()), indent=2))

## 4. Build object cutout library

This extracts object crops from COCO polygons. The generated-scene script can also build from COCO directly, but keeping a library is useful for inspection and debugging.

In [ ]:
object_library_cmd = [
    sys.executable,
    PIX2PIX_NOTEBOOK_DIR / "pix2pix_object_library.py",
    "--images_dir", RAW_IMAGES_DIR,
    "--coco_json", COCO_JSON,
    "--out_dir", OBJECT_LIBRARY_DIR,
    "--cat_to_palette_json", COLOR_PALETTE_JSON,
    "--pad", 8,
    "--min_area", 50,
    "--use_category_color",
    "--save_semantic_rgba",
    "--save_gray_crop",
    "--save_mask",
    "--save_preview",
]

run_cmd(object_library_cmd, cwd=PROJECT_ROOT, run=RUN_OBJECT_LIBRARY)
print(f"Preview crops after stage: {count_files(OBJECT_LIBRARY_DIR / 'preview')}")

## 5. Prepare empty trays and tray masks

Use these cells only when you are creating or refreshing tray masks. The tray mask editor opens a local OpenCV GUI.

In [ ]:
match_empty_cmd = [
    sys.executable,
    PIX2PIX_NOTEBOOK_DIR / "match_empty_trays.py",
    "--ab_dir", REAL_AB_TRAIN_DIR,
    "--empty_dir", EMPTY_TRAY_DIR,
    "--copy",
]
run_cmd(match_empty_cmd, cwd=PROJECT_ROOT, run=RUN_MATCH_EMPTY_TRAYS)

In [ ]:
tray_mask_cmd = [sys.executable, PIX2PIX_NOTEBOOK_DIR / "tray_mask.py"]
run_cmd(tray_mask_cmd, cwd=PROJECT_ROOT, run=RUN_TRAY_MASK_GUI, check=False)

## 6. Optional: build aligned AB dataset

`build_pix2pix_dataset.py` currently uses constants inside the script. Run this only when those constants point to the dataset you want to rebuild.

In [ ]:
build_dataset_cmd = [sys.executable, PIX2PIX_NOTEBOOK_DIR / "build_pix2pix_dataset.py"]
run_cmd(build_dataset_cmd, cwd=PROJECT_ROOT, run=RUN_BUILD_AB_DATASET)

## 7. Optional: train Pix2Pix

Use this when you need to train or refresh the checkpoint. The generation stage below assumes the checkpoint named in `MODEL_NAME` already exists.

In [ ]:
train_cmd = [
    sys.executable,
    PIX2PIX_EXTERNAL_DIR / "train.py",
    "--dataroot", REAL_AB_TRAIN_DIR.parent,
    "--name", MODEL_NAME,
    "--model", "pix2pix",
    "--direction", "AtoB",
    "--preprocess", "none",
    "--no_flip",
    "--load_size", RUN_CONFIG["canvas_h"],
    "--crop_size", RUN_CONFIG["canvas_w"],
    "--input_nc", 7,
    "--output_nc", 3,
    "--norm", "instance",
]

run_cmd(train_cmd, cwd=PROJECT_ROOT, run=RUN_TRAIN_PIX2PIX)

## 8. Generate synthetic scenes and score realism/novelty

This is the main production stage. It creates the temporary Pix2Pix test set, runs the checkpoint, exports generated images, and writes `generated_combo_summary.json` with Mahalanobis realism and nearest-real novelty scores.

In [ ]:
generate_cmd = [
    sys.executable,
    PIX2PIX_NOTEBOOK_DIR / "generate_pix2pixV2_MAHADIST.py",
    "--model_name", MODEL_NAME,
    "--generate_mode", RUN_CONFIG["generate_mode"],
    "--combo_mode", RUN_CONFIG["combo_mode"],
    "--images_dir", RAW_IMAGES_DIR,
    "--coco_json", COCO_JSON,
    "--classes", RUN_CONFIG["classes"],
    "--count", RUN_CONFIG["count"],
    "--blade_mask_dir", BLADE_MASK_TRAIN_DIR,
    "--tray_mask_dir", TRAY_MASK_TRAIN_DIR,
    "--tray_mask_thr", 0.5,
    "--tray_cc_close_px", 2,
    "--tray_mask_dilate_px", 0,
    "--rand_scale_min", RUN_CONFIG["rand_scale_min"],
    "--rand_scale_max", RUN_CONFIG["rand_scale_max"],
    "--rand_rot_min", RUN_CONFIG["rand_rot_min"],
    "--rand_rot_max", RUN_CONFIG["rand_rot_max"],
    "--horizontal_shift_only",
    "--max_horizontal_shift", RUN_CONFIG["max_horizontal_shift"],
    "--tray_horizontal_margin_px", RUN_CONFIG["tray_horizontal_margin_px"],
    "--tray_vertical_margin_px", RUN_CONFIG["tray_vertical_margin_px"],
    "--max_vertical_shift", RUN_CONFIG["max_vertical_shift"],
    "--x_search_step", RUN_CONFIG["x_search_step"],
    "--max_transform_candidates", RUN_CONFIG["max_transform_candidates"],
    "--out_dataset", OUT_DATASET,
    "--num_scenes", RUN_CONFIG["num_scenes"],
    "--seed", RUN_CONFIG["seed"],
    "--epoch", RUN_CONFIG["epoch"],
    "--real_eval_dir", REAL_EVAL_DIR,
    "--mahal_pca_dim", RUN_CONFIG["mahal_pca_dim"],
]

run_cmd(generate_cmd, cwd=PROJECT_ROOT, run=RUN_GENERATE_SCENES)
print(f"Generated summary expected at: {GENERATED_SUMMARY_JSON}")

## 9. Build held-out real thresholds and classify generated images

This compares generated scores to a held-out real distribution. It writes `real_vs_generated_report.json`.

In [ ]:
real_report_cmd = [
    sys.executable,
    PIX2PIX_NOTEBOOK_DIR / "realdataset_mahalanobis.py",
    "--real_ref_dir", REAL_REF_DIR,
    "--target_dir", REAL_EVAL_DIR,
    "--generated_summary_json", GENERATED_SUMMARY_JSON,
    "--out_json", REAL_VS_GENERATED_JSON,
    "--mahal_pca_dim", RUN_CONFIG["mahal_pca_dim"],
]

run_cmd(real_report_cmd, cwd=PROJECT_ROOT, run=RUN_REAL_VS_GENERATED_REPORT)
print(f"Real-vs-generated report expected at: {REAL_VS_GENERATED_JSON}")

## 10. Load reports into tables

In [ ]:
try:
    import pandas as pd
    HAS_PANDAS = True
except Exception as exc:
    pd = None
    HAS_PANDAS = False
    print(f"pandas is not available, skipping report tables: {exc}")

reports_available = GENERATED_SUMMARY_JSON.exists()
if not reports_available:
    print(f"Generated summary not found yet: {GENERATED_SUMMARY_JSON}")
    print("Run the generation stage with DRY_RUN = False, or point GENERATED_SUMMARY_JSON to an existing run.")
    generated_summary = {}
    real_report = None
else:
    generated_summary = json.loads(GENERATED_SUMMARY_JSON.read_text())
    real_report = json.loads(REAL_VS_GENERATED_JSON.read_text()) if REAL_VS_GENERATED_JSON.exists() else None

realism = generated_summary.get("realism_mahalanobis", {})
novelty = generated_summary.get("real_nn_novelty", {})

rows = []
for stem, rinfo in realism.get("per_image", {}).items():
    ninfo = novelty.get("per_image", {}).get(stem, {})
    rows.append({
        "scene": stem,
        "realism_mahalanobis": rinfo.get("mahalanobis"),
        "realism_status": rinfo.get("status"),
        "nearest_real_distance": ninfo.get("nearest_real_distance"),
        "nearest_real_stem": ninfo.get("nearest_real_stem"),
        "novelty_status": ninfo.get("status"),
    })

generated_columns = [
    "scene",
    "realism_mahalanobis",
    "realism_status",
    "nearest_real_distance",
    "nearest_real_stem",
    "novelty_status",
]
if HAS_PANDAS:
    generated_df = pd.DataFrame(rows, columns=generated_columns)
    if not generated_df.empty:
        generated_df = generated_df.sort_values("realism_mahalanobis", ascending=True, na_position="last")
else:
    generated_df = None
    print(f"Loaded {len(rows)} generated report rows.")

if HAS_PANDAS and real_report and real_report.get("comparison") and not generated_df.empty:
    class_df = pd.DataFrame.from_dict(
        real_report["comparison"]["generated_classification"], orient="index"
    ).rename_axis("scene").reset_index()
    generated_df = generated_df.merge(class_df[["scene", "classification"]], on="scene", how="left")

if HAS_PANDAS:
    display(generated_df.head(20))

if HAS_PANDAS and real_report:
    display(pd.concat([
        pd.Series(real_report["real_eval"]["stats"], name="held_out_real"),
        pd.Series(real_report["generated_eval"]["stats"], name="generated") if real_report.get("generated_eval") else pd.Series(name="generated"),
    ], axis=1))
    if real_report.get("comparison"):
        display(pd.Series(real_report["comparison"]["classification_counts"], name="count").to_frame())

## 11. Plot realism and novelty

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except Exception as exc:
    HAS_MATPLOTLIB = False
    plt = None
    print(f"matplotlib is not available, skipping plots: {exc}")

if not HAS_MATPLOTLIB:
    pass
elif generated_df is None:
    print("pandas is not available, so score plotting is skipped.")
elif generated_df.empty:
    print("No generated report loaded yet, so there is nothing to plot.")
else:
    plot_df = generated_df.dropna(subset=["realism_mahalanobis"])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(range(len(plot_df)), plot_df["realism_mahalanobis"].to_numpy(), marker="o", linewidth=1)
    axes[0].set_title("Generated realism score")
    axes[0].set_xlabel("ranked generated scene")
    axes[0].set_ylabel("Mahalanobis distance, lower is better")
    axes[0].grid(alpha=0.25)

    axes[1].scatter(plot_df["realism_mahalanobis"], plot_df["nearest_real_distance"], alpha=0.8)
    axes[1].set_title("Realism vs novelty")
    axes[1].set_xlabel("Mahalanobis realism, lower is better")
    axes[1].set_ylabel("Nearest-real distance, higher is more novel")
    axes[1].grid(alpha=0.25)

    if real_report and real_report.get("comparison"):
        th = real_report["comparison"]["thresholds"]
        axes[0].axhline(th["good_upper"], color="green", linestyle="--", label="real q95")
        axes[0].axhline(th["borderline_upper"], color="orange", linestyle="--", label="real q99")
        axes[0].legend()
        axes[1].axvline(th["good_upper"], color="green", linestyle="--", label="real q95")
        axes[1].axvline(th["borderline_upper"], color="orange", linestyle="--", label="real q99")
        axes[1].legend()

    plt.tight_layout()
    plt.show()

## 12. Preview best and worst generated scenes

In [ ]:
from PIL import Image

if "HAS_MATPLOTLIB" not in globals():
    try:
        import matplotlib.pyplot as plt
        HAS_MATPLOTLIB = True
    except Exception as exc:
        HAS_MATPLOTLIB = False
        plt = None
        print(f"matplotlib is not available, skipping previews: {exc}")

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def find_scene_image(scene):
    preferred = [
        f"{scene}_combo_smooth2x.png",
        f"{scene}_shampoo_smooth2x.png",
        f"{scene}_blade_smooth2x.png",
        f"{scene}_tray_smooth2x.png",
        f"{scene}.png",
        f"{scene}.jpg",
        f"{scene}.jpeg",
        f"{scene}.webp",
    ]
    for name in preferred:
        p = GENERATED_DIR / name
        if p.exists():
            return p
    if not GENERATED_DIR.exists():
        return None
    matches = sorted(p for p in GENERATED_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS and scene in p.name)
    return matches[0] if matches else None

def show_scene_grid(df, title, n=6):
    sample = df.dropna(subset=["realism_mahalanobis"]).head(n)
    if sample.empty:
        print(f"No images to show for: {title}")
        return
    fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3.7))
    if len(sample) == 1:
        axes = [axes]
    for ax, row in zip(axes, sample.itertuples(index=False)):
        ax.axis("off")
        img_path = find_scene_image(row.scene)
        label = getattr(row, "classification", "")
        if img_path is None:
            ax.set_title(f"{row.scene}\nmissing")
            continue
        ax.imshow(Image.open(img_path).convert("RGB"), cmap="gray")
        ax.set_title(f"{row.scene}\nM={row.realism_mahalanobis:.3f}\n{label}", fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

if not HAS_MATPLOTLIB:
    print("No matplotlib available, so image preview grids are skipped.")
elif generated_df is None:
    print("pandas is not available, so sorted image preview grids are skipped.")
elif generated_df.empty:
    print("No generated report loaded yet, so there are no scene previews.")
else:
    show_scene_grid(generated_df.sort_values("realism_mahalanobis", ascending=True), "Most real-like generated scenes")
    show_scene_grid(generated_df.sort_values("realism_mahalanobis", ascending=False), "Least real-like generated scenes")

## 13. Launch the interactive dashboard

This opens the Tkinter viewer when run from a local desktop Python environment.

In [ ]:
viewer_cmd = [
    sys.executable,
    PIX2PIX_NOTEBOOK_DIR / "mahal_scene_viewer.py",
    "--summary-json", GENERATED_SUMMARY_JSON,
    "--generated-images-dir", GENERATED_DIR,
]

print("$", " ".join(str(x) for x in viewer_cmd))
# subprocess.Popen([str(x) for x in viewer_cmd], cwd=PROJECT_ROOT)

## 14. Save run manifest

This writes a small JSON file beside the generated images so the run configuration is preserved.

In [ ]:
manifest = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "dry_run": DRY_RUN,
    "model_name": MODEL_NAME,
    "run_config": RUN_CONFIG,
    "paths": {k: str(v) for k, v in paths.items()},
}

manifest_path = GENERATED_DIR / "workflow_manifest.json"
if DRY_RUN:
    print(json.dumps(manifest, indent=2))
    print(f"DRY_RUN: manifest not written. Target would be: {manifest_path}")
else:
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"Saved manifest: {manifest_path}")